# Resource usage and inference latency

This notebook runs each local classifier benchmark as a separate process. Model and dataset loading are excluded from the measurements. Tokenization, device transfer, model inference, and probability conversion are included.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd
from IPython.display import display


def find_project_dir():
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "scripts" / "14_resource-latency").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the ai-detector-from-scratch project")


PROJECT_DIR = find_project_dir()
SCRIPT_DIR = PROJECT_DIR / "scripts" / "14_resource-latency"
RESULTS_DIR = SCRIPT_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR = SCRIPT_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = "auto"
BATCH_SIZE = 8
REPEATS = 5
LIMIT = None  # Set to 100 for a quick end-to-end check

BENCHMARK_SCRIPTS = [
    "benchmark_logreg.py",
    "benchmark_distilbert.py",
    "benchmark_distilbert_lora.py",
    "benchmark_distilbert_mica.py",
    "benchmark_modernbert.py",
    "benchmark_gpt2_fixed.py",
    "benchmark_gpt2_variable.py",
    "benchmark_qwen3_fixed.py",
    "benchmark_qwen3_variable.py",
]


In [ ]:
def run_benchmark(script_name):
    command = [
        sys.executable,
        str(SCRIPT_DIR / script_name),
        "--device",
        DEVICE,
        "--batch-size",
        str(BATCH_SIZE),
        "--repeats",
        str(REPEATS),
        "--json",
    ]
    if LIMIT is not None:
        command.extend(["--limit", str(LIMIT)])

    completed = subprocess.run(
        command,
        cwd=PROJECT_DIR,
        capture_output=True,
        text=True,
    )
    if completed.returncode != 0:
        raise RuntimeError(
            f"{script_name} failed with exit code {completed.returncode}:\n"
            f"{completed.stderr.strip()}"
        )
    output_lines = completed.stdout.strip().splitlines()
    if not output_lines:
        raise RuntimeError(f"{script_name} returned no output")
    return json.loads(output_lines[-1])


In [ ]:
results = []
for script_name in BENCHMARK_SCRIPTS:
    print(f"Running {script_name}...", flush=True)
    result = run_benchmark(script_name)
    results.append(result)
    print(
        f"  {result['mean_latency_ms_per_text']:.3f} ms/text, "
        f"{result['mean_texts_per_second']:.1f} texts/s"
    )


In [ ]:
results_frame = pd.DataFrame(results)
summary_columns = [
    "model",
    "device",
    "samples",
    "batch_size",
    "mean_latency_ms_per_text",
    "std_latency_ms_per_text",
    "mean_texts_per_second",
    "std_texts_per_second",
    "gpu_utilization_mean_percent",
    "gpu_utilization_peak_percent",
    "gpu_memory_allocated_peak_mb",
    "gpu_memory_reserved_peak_mb",
    "gpu_memory_used_peak_mb",
]
summary = results_frame[summary_columns].copy()
output_path = RESULTS_DIR / "resource-latency-results.csv"
summary.to_csv(output_path, index=False)
print(f"Saved {output_path}")
display(summary.round(3))
